# Part B (1/2) — Preprocessing, Cleaning & Feature Extraction (Flowers Recognition)

**Dataset:** [Flowers Recognition (Kaggle)](https://www.kaggle.com/datasets/alxmamaev/flowers-recognition) — raw images, 5 classes: daisy, dandelion, rose, sunflower, tulip.

**This notebook covers:** load raw images (skip non-image / corrupt files) → EDA → resize to one fixed size → normalize pixels to [0, 1] → encode labels → stratified 70/15/15 split → feature extraction (HOG + HSV color histogram + color stats) → `StandardScaler` + `PCA` (fit on **train only**) → **save the processed data**.


In [ ]:
import os, json, random, zipfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import joblib

warnings.filterwarnings('ignore')

# ---------- Reproducibility ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ---------- Config ----------
CLASSES = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']   # the 5 classes used
ZIP_PATH = '/content/flowers_dataset.zip'    # Colab: upload the Kaggle zip here
EXTRACT_PATH = '/content/dataset'

if os.path.exists(ZIP_PATH) and not os.path.exists(EXTRACT_PATH):
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)

def find_data_dir(root):
    # find the folder that directly contains the class sub-folders (works for any nesting)
    for dirpath, dirnames, _ in os.walk(root):
        if all(c in dirnames for c in CLASSES):
            return dirpath
    raise FileNotFoundError(f'Could not find class folders {CLASSES} under {root}')

DATA_DIR = os.environ.get('FLOWERS_DIR') or find_data_dir(EXTRACT_PATH)
print('Data dir:', DATA_DIR)

In [ ]:
# ---------- Shared feature code (also used by the GUI, so training and prediction match exactly) ----------
os.makedirs('part_b_images', exist_ok=True)
FEATURES_PY = '''
import cv2
import numpy as np
from PIL import Image
from skimage.feature import hog

IMG_SIZE = (128, 128)                     # one fixed size for every image
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(16, 16),
                  cells_per_block=(2, 2), block_norm='L2-Hys')
HSV_BINS = (8, 4, 4)                      # joint 3D HSV histogram -> 128 bins


def load_and_resize(src):
    # Open an image (path / file object / PIL image), convert to RGB, resize -> uint8 array
    img = Image.open(src) if not isinstance(src, Image.Image) else src
    img = img.convert('RGB').resize(IMG_SIZE, Image.LANCZOS)
    return np.asarray(img, dtype=np.uint8)


def normalize(x_uint8):
    # Scale pixel values from [0, 255] to [0, 1]
    return x_uint8.astype(np.float32) / 255.0


def extract_one(img_norm):
    # HOG (shape) + HSV color histogram (color) + per-channel mean/std for ONE normalized image
    u8 = (np.clip(img_norm, 0, 1) * 255).round().astype(np.uint8)
    gray = cv2.cvtColor(u8, cv2.COLOR_RGB2GRAY)
    hog_feat = hog(gray, **HOG_PARAMS)
    hsv = cv2.cvtColor(u8, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, list(HSV_BINS), [0, 180, 0, 256, 0, 256]).flatten()
    hist = hist / (hist.sum() + 1e-6)
    stats = np.concatenate([img_norm.mean(axis=(0, 1)), img_norm.std(axis=(0, 1))])
    return np.hstack([hog_feat, hist, stats]).astype(np.float32)


def extract_features(batch_uint8):
    # batch of resized uint8 images (N,H,W,3) -> normalize -> feature matrix (N, D)
    return np.stack([extract_one(normalize(im)) for im in batch_uint8])
'''
with open('part_b_images/image_features.py', 'w') as f:
    f.write(FEATURES_PY)

import sys
sys.path.insert(0, 'part_b_images')
import importlib, image_features
importlib.reload(image_features)
from image_features import IMG_SIZE, load_and_resize, normalize, extract_features, HOG_PARAMS, HSV_BINS
print('Image size after resizing:', IMG_SIZE)

## 1. Load data & EDA

In [ ]:
VALID_EXT = ('.jpg', '.jpeg', '.png', '.bmp')
X_list, y_list, paths, orig_sizes = [], [], [], []
corrupt = 0

for cls in CLASSES:
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in sorted(os.listdir(cls_dir)):
        if not fname.lower().endswith(VALID_EXT):
            continue                         # skip non-image files (e.g. .py files inside the Kaggle folders)
        fpath = os.path.join(cls_dir, fname)
        try:
            with Image.open(fpath) as im:
                orig_sizes.append(im.size)   # (width, height) BEFORE resizing
            X_list.append(load_and_resize(fpath))
            y_list.append(cls)
            paths.append(fpath)
        except Exception:
            corrupt += 1

X_uint8 = np.stack(X_list)                   # (N, 128, 128, 3) uint8
y_labels = np.array(y_list)
paths = np.array(paths)
del X_list
print('Loaded:', X_uint8.shape, '| corrupt/skipped files:', corrupt)

counts = pd.Series(y_labels).value_counts().reindex(CLASSES)
print(counts.to_string())

w = np.array([s[0] for s in orig_sizes]); h = np.array([s[1] for s in orig_sizes])
print(f'Original width  - min {w.min()}, max {w.max()}, mean {w.mean():.0f}')
print(f'Original height - min {h.min()}, max {h.max()}, mean {h.mean():.0f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], palette='viridis')
ax[0].set_title('Images per class'); ax[0].set_ylabel('count')
ax[1].hist(w, bins=30, alpha=.6, label='width'); ax[1].hist(h, bins=30, alpha=.6, label='height')
ax[1].set_title('Original image sizes'); ax[1].legend()
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(len(CLASSES), 5, figsize=(10, 2 * len(CLASSES)))
rng = np.random.RandomState(SEED)
for r, cls in enumerate(CLASSES):
    idx = rng.choice(np.where(y_labels == cls)[0], 5, replace=False)
    for c, i in enumerate(idx):
        axes[r, c].imshow(X_uint8[i]); axes[r, c].axis('off')
    axes[r, 0].set_title(cls, loc='left', fontsize=10)
plt.tight_layout(); plt.show()

## 2. Encode labels & stratified split (70% train / 15% validation / 15% test)

The split is done on the **raw resized images first**, so nothing fitted later (scaler, PCA) ever sees validation/test data.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y = le.fit_transform(y_labels)
print('Label mapping:', dict(zip(le.classes_, range(len(le.classes_)))))

idx_all = np.arange(len(y))
idx_temp, idx_test = train_test_split(idx_all, test_size=0.15, stratify=y, random_state=SEED)
idx_train, idx_val = train_test_split(idx_temp, test_size=0.15 / 0.85, stratify=y[idx_temp], random_state=SEED)

y_train, y_val, y_test = y[idx_train], y[idx_val], y[idx_test]
print(f'Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}')
print(pd.DataFrame({'train': np.bincount(y_train), 'val': np.bincount(y_val), 'test': np.bincount(y_test)},
                   index=le.classes_).to_string())

## 3. Normalization + feature extraction



In [ ]:
import time
t0 = time.time()
F_train = extract_features(X_uint8[idx_train])
F_val   = extract_features(X_uint8[idx_val])
F_test  = extract_features(X_uint8[idx_test])
print(f'Feature extraction took {time.time() - t0:.0f}s')

n_hog = F_train.shape[1] - (int(np.prod(HSV_BINS)) + 6)
print(f'Features per image: HOG={n_hog} + HSV hist={int(np.prod(HSV_BINS))} + color stats=6  ->  total {F_train.shape[1]}')
print('Feature matrices:', F_train.shape, F_val.shape, F_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Fit ONLY on training data
scaler = StandardScaler()
S_train = scaler.fit_transform(F_train)
S_val, S_test = scaler.transform(F_val), scaler.transform(F_test)

PCA_MAX = min(200, S_train.shape[0] - 1, S_train.shape[1])
pca = PCA(n_components=PCA_MAX, random_state=SEED)
P_train = pca.fit_transform(S_train)
P_val, P_test = pca.transform(S_val), pca.transform(S_test)

cum = np.cumsum(pca.explained_variance_ratio_)
print(f'PCA with {PCA_MAX} components keeps {cum[-1] * 100:.1f}% of the variance')
plt.plot(cum); plt.xlabel('# components'); plt.ylabel('cumulative explained variance')
plt.title('PCA (fit on train only)'); plt.grid(alpha=.3); plt.show()

## 5. Save the processed data (used by notebook 2)

In [ ]:
PROC_DIR = 'processed_part_b'
os.makedirs(PROC_DIR, exist_ok=True)

np.savez_compressed(
    os.path.join(PROC_DIR, 'arrays.npz'),
    S_train=S_train, S_val=S_val, S_test=S_test,          
    P_train=P_train, P_val=P_val, P_test=P_test,          
    y_train=y_train, y_val=y_val, y_test=y_test,
    X_test_uint8=X_uint8[idx_test],                      
    paths_test=paths[idx_test],
)
joblib.dump({'scaler': scaler, 'pca': pca, 'label_encoder': le}, os.path.join(PROC_DIR, 'objects.pkl'))
shutil.copy('part_b_images/image_features.py', os.path.join(PROC_DIR, 'image_features.py'))

meta = {
    'classes': list(le.classes_), 'img_size': list(IMG_SIZE), 'seed': SEED,
    'feature_dim': int(F_train.shape[1]), 'pca_max': int(PCA_MAX),
    'n_train': int(len(idx_train)), 'n_val': int(len(idx_val)), 'n_test': int(len(idx_test)),
}
with open(os.path.join(PROC_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved to', PROC_DIR, '->', sorted(os.listdir(PROC_DIR)))


try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree(PROC_DIR, '/content/drive/MyDrive/ml_project/part_b_processed', dirs_exist_ok=True)
    print('Copied to Google Drive: MyDrive/ml_project/part_b_processed')
except Exception as e:
    print('Drive copy skipped:', type(e).__name__)